## 1. Создание базовой ALS & Эмеддинги пользователей и товаров

In [ ]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20
)

als.fit(user_item_matrix)

user_embs = als.user_factors
item_embs = als.item_factors



## 2. Создание Teacher scores - данные для обучения CatBoost

In [ ]:
import numpy as np

def generate_dataset(interactions, user_embs, item_embs, n_neg=5):
    X = []
    y = []
    num_items = item_embs.shape[0]
    for user, pos_item in interactions:
        score = user_embs[user] @ item_embs[pos_item]
        X.append((user, pos_item))
        y.append(score)
        
        for _ in range(n_neg):
            neg_item = np.random.randint(0, num_items)

            score = user_embs[user] @ item_embs[neg_item]

            X.append((user, neg_item))
            y.append(score)
    y_norm = 1 / (1 + np.exp(-y))

    return X, y_norm



## 3. Выделение признаков пользователей и предметов

In [ ]:
def build_features(X, user_features, item_features):
    rows = []

    for user, item in X:
        uf = user_features[user]
        itf = item_features[item]

        row = np.concatenate([uf, itf])
        rows.append(row)

    return np.array(rows)

## 4. Обучаем CatBoost

In [ ]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=500,
    depth=4,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

model.fit(X_features, y)

## 5. Определение score для user and item

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))
    
def score_item(user_id, item_id):
    if item_popularity[item_id] >= WARM_THRESHOLD:
        # ALS
        return sigmoid(user_embs[user_id] @ item_embs[item_id])
    else:
        # CatBoost
        uf = user_features[user_id]
        itf = item_features[item_id]

        x = np.concatenate([uf, itf]).reshape(1, -1)
        return model.predict(x)[0]

## 6. Инференс

In [ ]:
def recommend(user_id, candidate_items):
    scores = []

    for item in candidate_items:
        s = score_item(user_id, item)
        scores.append(s)

    return sorted(
        zip(candidate_items, scores),
        key=lambda x: -x[1]
    )